In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [11]:
coff_eol = pd.read_parquet("processed_data/ONS/constrained_off_eolica_tm.parquet")

# NORDESTE AS TARGET
coff_eol = coff_eol.loc[coff_eol['nom_subsistema'] == "NORDESTE"]

In [12]:
coff_eol.columns

Index(['id_subsistema', 'nom_subsistema', 'id_estado', 'nom_estado',
       'nom_usina', 'id_ons', 'ceg', 'din_instante', 'val_geracao',
       'val_geracaolimitada', 'val_disponibilidade', 'val_geracaoreferencia',
       'val_geracaoreferenciafinal', 'cod_razaorestricao',
       'cod_origemrestricao', 'dsc_restricao', 'fonte', 'ano', 'mes', 'data',
       'hora', 'minuto', 'ano_mes', 'arquivo_origem'],
      dtype='str')

In [13]:
coff_estados = coff_eol.groupby(["id_estado", "din_instante"]).agg(
    {
        "val_geracao": "sum",
        "val_geracaolimitada": "sum",
        "val_disponibilidade": "sum"
    }
).reset_index()

In [14]:
coff_estados["id_estado"].unique()

<ArrowStringArray>
['BA', 'CE', 'PB', 'PE', 'PI', 'RN']
Length: 6, dtype: str

In [18]:
estado = "BA"
coff_estados["din_instante"] = pd.to_datetime(coff_estados["din_instante"])
df_estado = (
    coff_estados.loc[coff_estados["id_estado"].eq(estado)]
    .sort_values("din_instante")
)

fig = go.Figure()
fig.add_scatter(
    x=df_estado["din_instante"],
    y=df_estado["val_geracao"],
    mode="lines",
    name="Geracao",
)
fig.add_scatter(
    x=df_estado["din_instante"],
    y=df_estado["val_geracaolimitada"],
    mode="lines",
    name="Geracao Limitada",
)

fig.update_layout(
    title=f"Dados do estado {estado}",
    xaxis_title="Data",
    yaxis_title="Valores",
    hovermode="x unified",
)
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=7, label="1 semana", step="day", stepmode="backward"),
            dict(count=1, label="1 mes", step="month", stepmode="backward"),
            dict(count=6, label="6 meses", step="month", stepmode="backward"),
            dict(count=1, label="1 ano", step="year", stepmode="backward"),
            dict(label="Tudo", step="all"),
        ]
    ),
)
fig.show()